In [1]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["USE_TF"] = "0"   


In [2]:
import pandas as pd

FILES = [
    "election2024_tweets.csv",
    "election2020_tweets.csv",
    "Capitol_Attack202_tweets.csv",
    "covid_tweets3.csv",
    "Summer2020_tweets2.csv",
    "CapitolTel.csv",
    "Elec2024Tel.csv",
    "Utah_tweets2.csv",
    "Roe2022_tweets.csv",
    "midterm_tweets224.csv"
    
]


TEXT_COLUMN = "Text"   # change if your column name is different
OUTFILE = "combined.csv"

first_write = True

for file in FILES:
    for chunk in pd.read_csv(file, chunksize=50000):

        # keep only the text column
        chunk = chunk[[TEXT_COLUMN]]

        # drop empty rows
        chunk = chunk.dropna()

        # append to output
        chunk.to_csv(
            OUTFILE,
            mode="w" if first_write else "a",
            index=False,
            header=first_write
        )

        first_write = False

print("✅ Done! Combined text file created.")


✅ Done! Combined text file created.


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
print("✅ transformers imported without TF")


/home/gg2713/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ transformers imported without TF


In [4]:
import pandas as pd

df = pd.read_csv('combined.csv')

df.head()
df.count()

Text    470815
dtype: int64

In [5]:
import os
import pandas as pd

FILES = [
    "blm_toxic_textonly.csv",
    "capitol_attack_toxic_synthetic.csv",
    "covid_toxic_textonly.csv",
    "election2020_toxic_synthetic.csv",
    "election2024_ai_generated.csv",
    "roe_wade_toxic_textonly.csv",
    "utah_shooting_toxic_textonly.csv",
    "Utah_Shooting_2025_Dataset.csv",
    "Roe2022_synthetic_posts.csv",
    "Election2024_synthetic_posts.csv",
    "Capitol_Attack_synthetic_posts.csv",
    "Capitol_Attack_synthetic_posts1.csv",
    "Midterm_synthetic_posts.csv",
    "Summer2020_synthetic_posts.csv",
    "Covid_synthetic_posts.csv"
]


TEXT_COLUMN = "Text"   # change if your column name is different
OUTFILE = "combined_AI.csv"

first_write = True

for file in FILES:
    for chunk in pd.read_csv(file, chunksize=50000):

        # keep only the text column
        chunk = chunk[[TEXT_COLUMN]]

        # drop empty rows
        chunk = chunk.dropna()

        # append to output
        chunk.to_csv(
            OUTFILE,
            mode="w" if first_write else "a",
            index=False,
            header=first_write
        )

        first_write = False

print("✅ Done! Combined text file created.")


✅ Done! Combined text file created.


In [6]:
import sys, site, importlib.util

print("Python:", sys.executable)
print("User site:", site.getusersitepackages())
print("User site in sys.path?", site.getusersitepackages() in sys.path)

print("find_spec('sentencepiece') =", importlib.util.find_spec("sentencepiece"))

try:
    import sentencepiece
    print("✅ sentencepiece imported:", sentencepiece.__version__)
    print("sentencepiece file:", sentencepiece.__file__)
except Exception as e:
    print("❌ sentencepiece import failed:", e)

print("\nFirst 5 sys.path entries:\n", "\n".join(sys.path[:5]))



Python: /share/apps/NYUAD5/miniconda/3-4.11.0/envs/jupyter/bin/python
User site: /home/gg2713/.local/lib/python3.12/site-packages
User site in sys.path? True
find_spec('sentencepiece') = ModuleSpec(name='sentencepiece', loader=<_frozen_importlib_external.SourceFileLoader object at 0x1553b42cf8c0>, origin='/home/gg2713/.local/lib/python3.12/site-packages/sentencepiece/__init__.py', submodule_search_locations=['/home/gg2713/.local/lib/python3.12/site-packages/sentencepiece'])
✅ sentencepiece imported: 0.2.1
sentencepiece file: /home/gg2713/.local/lib/python3.12/site-packages/sentencepiece/__init__.py

First 5 sys.path entries:
 /share/apps/NYUAD5/miniconda/3-4.11.0/envs/jupyter/lib/python312.zip
/share/apps/NYUAD5/miniconda/3-4.11.0/envs/jupyter/lib/python3.12
/share/apps/NYUAD5/miniconda/3-4.11.0/envs/jupyter/lib/python3.12/lib-dynload

/home/gg2713/.local/lib/python3.12/site-packages


In [7]:
import site, sys
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.append(user_site)

import sentencepiece
print("✅ sentencepiece:", sentencepiece.__version__)



✅ sentencepiece: 0.2.1


In [8]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base", use_fast=False)
print("✅ tokenizer loaded")


✅ tokenizer loaded


In [9]:
import os, numpy as np, pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import evaluate

HUMAN_FILE = "combined.csv"
AI_FILE    = "combined_AI_clean.csv"
TEXT_COL   = "Text"

MODEL_NAME = "microsoft/deberta-v3-base"
OUT_DIR    = "./deberta_detector_fixed"

MAX_LEN = 128
SEED = 42
SAMPLE_PER_CLASS = 400000  # keep your balanced sample for now
USE_FP16 = False          # IMPORTANT: stop nan loss

def load_text_only(path: str, label: int) -> pd.DataFrame:
    df = pd.read_csv(path, usecols=[TEXT_COL]).dropna()
    df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
    df = df[df[TEXT_COL] != ""]
    df = df.drop_duplicates(subset=[TEXT_COL])
    df["labels"] = label  # IMPORTANT: name it labels explicitly
    print(df["labels"].value_counts())
    if SAMPLE_PER_CLASS is not None and len(df) > SAMPLE_PER_CLASS:
        df = df.sample(SAMPLE_PER_CLASS, random_state=SEED)
    return df.reset_index(drop=True)

human = load_text_only(HUMAN_FILE, 0)
ai    = load_text_only(AI_FILE, 1)

df = pd.concat([human, ai], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
print("Rows:", len(df))
print(df["labels"].value_counts())

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df["labels"])
val_df, test_df   = train_test_split(temp_df, test_size=0.5, random_state=SEED, stratify=temp_df["labels"])

ds = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False),
    "test": Dataset.from_pandas(test_df, preserve_index=False),
})

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def tokenize(batch):
    return tokenizer(batch[TEXT_COL], truncation=True, max_length=MAX_LEN)

# remove only the text col; keep "labels"
ds_tok = ds.map(tokenize, batched=True, remove_columns=[TEXT_COL])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Force clean load; if your HF cache is corrupted, this helps:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
)

# ---- metrics
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "precision": precision.compute(predictions=preds, references=labels, average="binary")["precision"],
        "recall": recall.compute(predictions=preds, references=labels, average="binary")["recall"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

# ---- training args: stable defaults
args = TrainingArguments(
    output_dir=OUT_DIR,
    num_train_epochs=2,
    learning_rate=1e-5,                 # slightly safer than 2e-5
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    warmup_steps=500,                   # replace deprecated warmup_ratio
    logging_steps=200,
    save_strategy="epoch",
    eval_strategy="epoch" if "eval_strategy" in TrainingArguments.__init__.__code__.co_varnames else "epoch",
    fp16=USE_FP16,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# ---- sanity check: loss should be ~0.69 BEFORE training, not 0 / nan
batch = next(iter(torch.utils.data.DataLoader(ds_tok["train"], batch_size=8, collate_fn=data_collator)))
batch = {k: v.to(model.device) for k, v in batch.items()}
with torch.no_grad():
    out = model(**batch)
print("Sanity loss (should be finite):", float(out.loss))

trainer.train()

val_metrics = trainer.evaluate(ds_tok["validation"])
print("\nValidation metrics:", val_metrics)

test_out = trainer.predict(ds_tok["test"])
test_preds = np.argmax(test_out.predictions, axis=-1)
test_labels = ds_tok["test"]["labels"]

print("\nConfusion Matrix:\n", confusion_matrix(test_labels, test_preds))
print("\nClassification Report:\n", classification_report(test_labels, test_preds, digits=4))

trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print("✅ Saved to", OUT_DIR)


labels
0    424152
Name: count, dtype: int64
labels
1    246481
Name: count, dtype: int64
Rows: 646481
labels
0    400000
1    246481
Name: count, dtype: int64


Map: 100%|██████████| 64649/64649 [00:14<00:00, 4512.48 examples/s]
Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Sanity loss (should be finite): 0.7329966425895691


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.000000,0.000992,0.999845,0.999676,0.999919,0.999797
2,0.002300,0.001412,0.999814,0.999554,0.999959,0.999757



Validation metrics: {'eval_loss': 0.001411515404470265, 'eval_accuracy': 0.999814379408489, 'eval_precision': 0.9995538973152729, 'eval_recall': 0.9999594287568971, 'eval_f1': 0.9997566219121405, 'eval_runtime': 402.3007, 'eval_samples_per_second': 160.696, 'eval_steps_per_second': 5.024, 'epoch': 2.0}

Confusion Matrix:
 [[39989    11]
 [    0 24649]]

Classification Report:
               precision    recall  f1-score   support

           0     1.0000    0.9997    0.9999     40000
           1     0.9996    1.0000    0.9998     24649

    accuracy                         0.9998     64649
   macro avg     0.9998    0.9999    0.9998     64649
weighted avg     0.9998    0.9998    0.9998     64649

✅ Saved to ./deberta_detector_fixed
